In [1]:
import pandas as pd
import numpy as np
import os
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler


In [7]:
folder_path = "flight_data_24_25_pr_fe"

csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

dataframes = {}

for file in csv_files:
    path = os.path.join(folder_path, file)
    dataframes[file] = pd.read_csv(path)

print(dataframes.keys())

dict_keys(['test_pre_fe.csv', 'train_pre_fe.csv', 'val_pre_fe.csv'])


In [9]:
train = dataframes['train_pre_fe.csv']
val = dataframes['val_pre_fe.csv']
test = dataframes['test_pre_fe.csv']

In [ ]:
def add_features_optimized(df):
    
    # Tạo feature Mùa (season) từ tháng
    month = df['month'].values
    season_conditions = [
        np.isin(month, [12, 1, 2]), # Winter
        np.isin(month, [3, 4, 5]),  # Spring
        np.isin(month, [6, 7, 8])   # Summer
    ]
    season_choices = [0, 1, 2] # 3 là mặc định cho Fall
    df['season'] = np.select(season_conditions, season_choices, default=3).astype(np.int8)

    # Tạo feature Buổi (time_of_day) từ giờ
    hour = df['hour'].values
    tod_conditions = [
        (hour >= 5) & (hour < 12),  # Morning
        (hour >= 12) & (hour < 17), # Afternoon
        (hour >= 17) & (hour < 21)  # Evening
    ]
    tod_choices = [0, 1, 2] # 3 là mặc định cho Night
    df['time_of_day'] = np.select(tod_conditions, tod_choices, default=3).astype(np.int8)

    return df

gc.collect()
train = add_features_optimized(train)

gc.collect()
val = add_features_optimized(val)

gc.collect()
test = add_features_optimized(test)

gc.collect()


0

In [11]:

def create_weather_features(train_df, val_df, test_df):
    dfs = [train_df, val_df, test_df]
    names = ['train', 'val', 'test']
    
    scaler = MinMaxScaler()
    cols_to_scale = ['wind_speed_10m', 'cloud_cover']
    
    train_df[['wind_speed_norm', 'cloud_cover_norm']] = scaler.fit_transform(train_df[cols_to_scale])
    val_df[['wind_speed_norm', 'cloud_cover_norm']] = scaler.transform(val_df[cols_to_scale])
    test_df[['wind_speed_norm', 'cloud_cover_norm']] = scaler.transform(test_df[cols_to_scale])

    # ÁP DỤNG CHO TỪNG TẬP BẰNG VECTORIZATION
    for df, name in zip(dfs, names):
        
        is_storm = df.get('weather_Storm', 0) == 1
        is_snow = df.get('weather_Snow_Ice', 0) == 1
        is_fog = df.get('weather_Fog', 0) == 1
        is_rain = df.get('weather_Rain', 0) == 1
        
        wind_speed = df.get('wind_speed_10m', 0)
        cloud_cover = df.get('cloud_cover', 0)

        cond_storm = is_storm | ((is_rain | is_snow) & (wind_speed > 30))
        cond_snow_fog = is_snow | is_fog
        cond_rain = is_rain
        cond_cloudy = cloud_cover > 50 
        
        conditions = [cond_storm, cond_snow_fog, cond_rain, cond_cloudy]
        choices = [4, 3, 2, 1]
        
        df['weather_severity_score'] = np.select(conditions, choices, default=0)
        
        df['is_bad_weather'] = (df['weather_severity_score'] >= 2).astype(int)
        
        df['wind_risk'] = df['wind_speed_norm'] * df['cloud_cover_norm']
        
        df.drop(columns=['wind_speed_norm', 'cloud_cover_norm'], inplace=True, errors='ignore')
        

    return train_df, val_df, test_df

train, val, test = create_weather_features(train, val, test)

In [ ]:
def create_interaction_features(train_df, val_df, test_df, 
                                distance_col='distance', 
                                wind_col='wind_speed_10m', 
                                weather_col='weather_severity_score'): 
    
    train = train_df.copy()
    val = val_df.copy()
    test = test_df.copy()
    
    for df in [train, val, test]:
        # TƯƠNG TÁC: GIỜ CAO ĐIỂM x THỜI TIẾT (hour_x_weather)
        rush_hours = [6, 7, 8, 10, 11, 17, 18]        
        # Tạo cờ (flag) giờ cao điểm: Trả về 1 nếu là giờ cao điểm, 0 nếu giờ bình thường
        df['is_rush_hour'] = df['hour'].isin(rush_hours).astype(int)
        
        # Nhân cờ này với độ khắc nghiệt thời tiết
        # Kết quả: Bão vào giờ cao điểm = Điểm rất cao; Bão giờ thường = Điểm thấp hơn
        df['rush_hour_x_weather'] = df['is_rush_hour'] * df[weather_col]
        
        # Tương tác đa thức bậc 2 để làm nổi bật tác động kép
        df['weather_impact_squared'] = df['is_rush_hour'] * (df[weather_col] ** 2)

        # TƯƠNG TÁC: QUÃNG ĐƯỜNG x TỐC ĐỘ GIÓ (distance_x_wind)
        # df['distance_x_wind'] = df[distance_col] * df[wind_col]
        df['wind_per_mile'] = df[wind_col] / (df[distance_col] + 1)

        df.drop(columns=['is_rush_hour'], inplace=True)
        
    return train, val, test


train, val, test = create_interaction_features(
    train, val, test, 
    distance_col='distance', 
    wind_col='wind_speed_10m',        
    weather_col='weather_severity_score'   
)

In [17]:
BASE_DIR = 'flight_data_24_25'
os.makedirs(BASE_DIR, exist_ok=True)
train.to_csv(os.path.join(BASE_DIR, 'train.csv'), index=False)
val.to_csv(os.path.join(BASE_DIR, 'valid.csv'), index=False)
test.to_csv(os.path.join(BASE_DIR, 'test.csv'), index=False)